### Multiclass Classification
 : 결과가 여러개의 클래스로 나오는 경우 

## Wine Recognition Dataset
 : 와인의 13가지 특징으로 3개의 품종중 어디에 속하는지 다중분류

- 데이터 개수 : 178개 
- 입력 특징 : 13개 
- 정답 클래스 : 3개(1,2,3 중 하나)

In [3]:
import os 
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt 
import torch
from torch import nn, optim
from torch.utils.data import TensorDataset, DataLoader 

torch.manual_seed(42)

#### Step 1. 데이터 불러오기 및 전처리 

In [ ]:
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/wine/wine.data'

columns = ['class',	
            'Alcohol',
            'Malicacid',	
            'Ash',	
            'Alcalinity_of_ash',	
            'Magnesium',	
            'Total_phenols',	
            'Flavanoids',	
            'Nonflavanoid_phenols',	
            'Proanthocyanins',	
            'Color_intensity',	
            'Hue',
            '0D280_0D315_of_diluted_wines',				
            'Proline'
            ]

df = pd.read_csv(url, names = columns)
# print(df.head())
# print('데이터 크기 :', df.shape)
# print('클래스별 데이터 개수 :', df["class"].value_counts().sort_index())

# 입력(X), 출력(y) 분리
x = columns[1:]
X = torch.tensor(df[x].values, dtype = torch.float32)


# 원본 클래스 1,2,3 -> 0,1,2 변환 
y = df['class'].values-1
Y = torch.tensor(y, dtype = torch.float32)

# training(80%), test(20%) 로 분할 
train_indices = []
test_indices = []

for class_id in [0,1,2] : 
    class_indices = np.where(Y == class_id)[0]
    class_indices = torch.randperm(len(class_indices)).numpy()

    split = int(len(class_indices)* 0.8)
    train_indices.extend(class_indices[:split].tolist())
    test_indices.extend(class_indices[split:].tolist())


# 학습/테스트 인덱스 섞기 
train_indices = torch.tensor(train_indices)[torch.randperm(len(train_indices))]
test_indices = torch.tensor(test_indices)[torch.randperm(len(test_indices))]

# 원본데이터 분리 (train입력, train출력, test 입력, test 출력)
X_train = X[train_indices]
X_test = X[test_indices]
Y_train = Y[train_indices]
Y_test = Y[test_indices]

# 표준화 
mean = X_train.mean(dim=0)
std = X_train.std(dim=0). clamp(min = 1e-5)

X_train_scaled = (X_train - mean) / std
X_test_scaled = (X_test - mean) / std

Y_train = torch.tensor(Y_train, dtype = torch.long)
Y_test = torch.tensor(Y_test, dtype = torch.long)

# 오분류 사례를 원래 단위로 확인하기 위한 테스트 데이터 보관
test_df = df.iloc[test_indices.numpy()].reset_index(drop = True)
print(test_df)

print('X_train :', X_train.shape, X_train.dtype)
print('Y_train :', Y_train.shape, Y_train.dtype)
print('X_test :', X_test.shape, X_test.dtype)
print('Y_test :', Y_test.shape, Y_test.dtype)

print('학습 데이터 클래스 개수 :', torch.bincount(Y_train, minlength=3))
print('테스트 데이터 클래스 개수 :', torch.bincount(Y_test, minlength=3))
# bincount => 0이상의 자연수를 몇번 넣었는지 세주는 메서드




    class  Alcohol  Malicacid   Ash  Alcalinity_of_ash  Magnesium  \
0       1    13.74       1.67  2.25               16.4        118   
1       1    13.86       1.35  2.27               16.0         98   
2       2    12.37       1.21  2.56               18.1         98   
3       1    13.73       1.50  2.70               22.5        101   
4       1    13.20       1.78  2.14               11.2        100   
5       1    14.30       1.92  2.72               20.0        120   
6       1    14.02       1.68  2.21               16.0         96   
7       1    13.16       2.36  2.67               18.6        101   
8       1    13.39       1.77  2.62               16.1         93   
9       1    13.56       1.71  2.31               16.2        117   
10      1    13.05       1.65  2.55               18.0         98   
11      1    14.38       1.87  2.38               12.0        102   
12      2    12.37       1.13  2.16               19.0         87   
13      1    13.20       1.78  2.1

C:\Users\mega\AppData\Local\Temp\ipykernel_13908\1073207765.py:63: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  Y_train = torch.tensor(Y_train, dtype = torch.long)
C:\Users\mega\AppData\Local\Temp\ipykernel_13908\1073207765.py:64: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  Y_test = torch.tensor(Y_test, dtype = torch.long)


### Step 2. DataLoader 생성 

In [28]:
BATCH_SIZE = 16

train_DS = TensorDataset(X_train, Y_train)
test_DS = TensorDataset(X_test, Y_test)

train_DL = DataLoader(train_DS, batch_size = BATCH_SIZE, shuffle = True)
test_DL = DataLoader(test_DS, batch_size = BATCH_SIZE, shuffle = False)

x_batch, y_batch = next(iter(train_DL))
print('입력 배치 shape', x_batch.shape)
print('정답 배치 shape', y_batch.shape)

입력 배치 shape torch.Size([16, 13])
정답 배치 shape torch.Size([16])


### Step 3. 학습 모델

In [29]:
class MLP(nn.Module) : 
    def __init__(self) : 
        super().__init__()

        self.linear = nn.Sequential(nn.Linear(13, 30), 
                                    nn.ReLU(),
                                    nn.Linear(30, 3))
    def forward(self, x) :
        return self.linear(x)

model = MLP()

### Step 4. 모델 학습시키기

In [ ]:
LR = 1e-3
EPOCH = 100

optimizer = optim.Adam(model.parameters(), lr = LR)
criterion = nn.CrossEntropyLoss()

loss_history = []
N = len(train_DL.dataset)

save_path = 'results/MLP_Wine.pt'

if os.path.exists(save_path) : 
    model.load_state_dict(torch.load(save_path))
    print('기존에 저장된 모델을 불러왔습니다')
else :
    model.train()
    for ep in range(EPOCH) : 
        rloss = 0
        for x_batch, y_batch in train_DL : 
            # 예측
            y_hat = model(x_batch)
            # loss
            loss = criterion(y_hat, y_batch)

            # update
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            rloss += loss.item() * x_batch.shape[0]

        epoch_loss = rloss / N 
        loss_history.append(epoch_loss)